# 02 · Cleaning

Each decision here points at a number from notebook 01. The production
implementation is `src/cleaning.py`; this notebook shows the reasoning and
verifies the result.

Nothing is silently deleted: rows that cannot be scored go to a quarantine file
with a reason code.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
                     "axes.edgecolor": "#e3e2dc", "grid.color": "#e3e2dc",
                     "axes.titleweight": "bold", "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False})
S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"

In [2]:
import cleaning

raw = pd.read_csv(config.RAW_TRANSACTIONS)
clean, quarantine, log = cleaning.clean(verbose=False)
pd.Series(log).to_frame("count")

,count
rows_in,60154
unparsed_timestamps,0
sender_country_imputed,601
channel_unknown,1203
exact_duplicates_dropped,14
near_duplicates_dropped,1
ids_still_duplicated,0
wire_transfer_type_remapped,39
quarantined_missing_amount,60
quarantined_negative_amount,20


## 1. Timestamps, parsed per format

Each format is parsed explicitly rather than inferred, for the reason found in
notebook 01.

In [3]:
demo = pd.Series(["2025-03-08 10:40:21", "30/04/2025 00:13", "03/04/2025 09:00"])
parsed = cleaning.parse_timestamps(demo)
pd.DataFrame({"raw": demo, "parsed": parsed, "month": parsed.dt.month})

,raw,parsed,month
0,2025-03-08 10:40:21,2025-03-08 10:40:21,3
1,30/04/2025 00:13,2025-04-30 00:13:00,4
2,03/04/2025 09:00,2025-04-03 09:00:00,4


`03/04/2025` becomes **3 April**, not 4 March. All 60,154 rows parse; zero `NaT`.

## 2. De-duplication, in two passes

Pass A removes byte-identical rows. Pass B catches the replay that differs only
in seconds precision, matching on a business key.

In [4]:
print(f"exact duplicates dropped : {log['exact_duplicates_dropped']}")
print(f"near duplicates dropped  : {log['near_duplicates_dropped']}")
print(f"duplicate ids remaining  : {clean.transaction_id.duplicated().sum()}")

exact duplicates dropped : 14
near duplicates dropped  : 1
duplicate ids remaining  : 0


Replayed debits would otherwise inflate account velocity and manufacture
structuring alerts on their own.

## 3. Missing countries, imputed and validated

Before imputing, test the assumption: where `sender_country` is present, does it
match the account's home country?

In [5]:
cust = pd.read_csv(config.CUSTOMERS).set_index("account_id")["home_country"]
std = cleaning.standardise_country(raw.sender_country)
known = std.notna()
agreement = (std[known].values == raw.loc[known, "sender_account"].map(cust).values).mean()
print(f"sender_country == account home_country: {agreement:.2%}")
print(f"rows imputed from the customer file   : {log['sender_country_imputed']}")

sender_country == account home_country: 99.79%
rows imputed from the customer file   : 601


99.79% agreement makes the customer file a reliable source for the 601 gaps.

`channel` is handled differently: 1,203 missing values become `"unknown"`, its
own category. A failed feed is information, and there is no second source to
recover the true value from.

## 4. Country spellings folded onto canonical names

In [6]:
before = pd.concat([raw.sender_country, raw.receiver_country]).nunique()
after = pd.concat([clean.sender_country, clean.receiver_country]).nunique()
print(f"distinct country strings: {before} -> {after}")
sorted(clean.sender_country.unique())

distinct country strings: 22 -> 14


['Australia',
 'Canada',
 'Cayman Islands',
 'Cyprus',
 'France',
 'Germany',
 'Ireland',
 'Malta',
 'Netherlands',
 'Panama',
 'Seychelles',
 'Spain',
 'United Kingdom',
 'United States']

## 5. Unscoreable rows quarantined, not deleted

60 rows have no amount and 20 are negative. Neither can be scored by an
amount-based rule.

In [7]:
print(quarantine.quarantine_reason.value_counts().to_string())

truth = (pd.read_csv(config.GROUND_TRUTH).drop_duplicates("transaction_id")
           .set_index("transaction_id")["is_laundering_pattern"])
lost = quarantine.transaction_id.map(truth).fillna(0).sum()
print(f"\nknown cases lost to quarantine: {int(lost)}")
print("recall ceiling stays at 100%" if lost == 0 else "RECALL CEILING REDUCED")

quarantine_reason
amount_missing     60
amount_negative    20

known cases lost to quarantine: 0
recall ceiling stays at 100%


Worth checking explicitly: anything cleaning throws away is a case no
downstream layer can ever win back.

## 6. Both leakage paths closed

The ID is kept as a reference for the case list but never reaches a feature
matrix. The stray `wire_transfer` type is folded into `transfer`, which is what
it actually is.

In [8]:
print(f"wire_transfer types remapped: {log['wire_transfer_type_remapped']}")
print(f"transaction_type values now : {sorted(clean.transaction_type.unique())}")
print(f"still valid as a channel    : {'wire_transfer' in set(clean.channel)}")

wire_transfer types remapped: 39
transaction_type values now : ['deposit', 'payment', 'transfer', 'withdrawal']
still valid as a channel    : True


## Result

In [9]:
print(f"{log['rows_in']:,} raw -> {log['rows_out']:,} clean, {len(quarantine)} quarantined")
clean.head(3)

60,154 raw -> 60,059 clean, 80 quarantined


,transaction_id,timestamp,sender_account,receiver_account,amount,currency,sender_country,receiver_country,transaction_type,channel,is_cross_border,hour,day_of_week,date
0,TXN0038984,2025-01-01 00:01:25,ACC102183,ACC101952,674.56,GBP,Canada,Netherlands,deposit,online_transfer,1,0,2,2025-01-01
1,TXN0032901,2025-01-01 00:11:52,ACC102577,ACC100681,139.83,GBP,United States,Malta,deposit,wire_transfer,1,0,2,2025-01-01
2,TXN0048264,2025-01-01 00:21:57,ACC100378,ACC101184,200.15,GBP,Seychelles,United States,payment,wire_transfer,1,0,2,2025-01-01
